# QLoRA Supervised Fine-Tuning — Llama-3-8B-Instruct (TechQA)

**Διπλωματικη εργασια — Φαση 2: Learned Abstention & Grounded Resolution**

Το notebook εκπαιδευει το `Meta-Llama-3-8B-Instruct` με **4-bit QLoRA** πανω στο SFT dataset
(450 παραδειγματα, ~55% solve / ~45% abstain) που παρηχθη απο το RAG pipeline. Στοχος **δεν** ειναι
νεα γνωση αλλα **συμπεριφορα**: σταθερο format (Root cause → βηματα → `Technote: [id]`),
πειθαρχια αποχης (MODE B), λιγοτερα εφευρεμενα IDs — δηλαδη ακριβως το *hazard* που το
heuristic retrieval gate δεν μπορουσε να λυσει.

**Ροη:** setup → φορτωση dataset → επαληθευση μηκους & masking → 4-bit load + LoRA → training →
merge → GGUF/Q4_K_M → sanity check → οδηγιες για Ollama + τριπλη αξιολογηση.

**Απαιτησεις Kaggle:** Accelerator = **GPU T4** (οχι P100). Επισυναψε το `sft_dataset.jsonl`
ως Kaggle Dataset και προσαρμοσε το `DATASET_PATH` παρακατω.

> **Reproducibility:** το πρωτο code cell τυπωνει τις εκδοσεις των βιβλιοθηκων — **καταγραψε τες**
> στο παραρτημα της διπλωματικης.

## 1. Εγκατασταση βιβλιοθηκων

In [ ]:
# Ενημερωμενο HF stack. (Το -U φερνει τις τρεχουσες εκδοσεις· τυπωνονται στο επομενο cell.)
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets
# Το Kaggle εχει προεγκατεστημενο παλιο torchao (0.10) που κανει το PEFT να σκαει στο merge.
# Δεν το χρησιμοποιουμε (bitsandbytes 4-bit) — το αφαιρουμε.
!pip uninstall -y torchao

In [ ]:
import os
# Μειωση fragmentation στη GPU μνημη — ΠΡΕΠΕΙ να οριστει πριν το import torch.
# (Αν το εχεις ηδη τρεξει: Restart kernel -> Run All, αλλιως δεν ενεργοποιειται.)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch, transformers, trl, peft, bitsandbytes, accelerate, datasets

print("torch        ", torch.__version__)
print("transformers ", transformers.__version__)
print("trl          ", trl.__version__)
print("peft         ", peft.__version__)
print("bitsandbytes ", bitsandbytes.__version__)
print("accelerate   ", accelerate.__version__)
print("datasets     ", datasets.__version__)

assert torch.cuda.is_available(), "Δεν βρεθηκε GPU — Settings > Accelerator > GPU T4"
gpu_name = torch.cuda.get_device_name(0)
print("\nGPU:", gpu_name)
print("bf16 supported:", torch.cuda.is_bf16_supported())
if "P100" in gpu_name:
    print("\n[ΠΡΟΣΟΧΗ] P100 (sm_60): πιθανα προβληματα με νεες βιβλιοθηκες. Προτιμησε T4.")
# ΣΗΜΕΙΩΣΗ: καταγραψε τις παραπανω εκδοσεις στο παραρτημα της διπλωματικης.

## 2. Παραμετροι

In [ ]:
# --- Μοντελο & δεδομενα ---
MODEL_NAME   = "NousResearch/Meta-Llama-3-8B-Instruct"   # ungated mirror — ΙΔΙΑ βαρη με το served μοντελο (llama3:8b-instruct)
DATASET_PATH = "/kaggle/input/techqa-sft/sft_dataset.jsonl"  # <-- ΠΡΟΣΑΡΜΟΣΕ στο Kaggle Dataset σου

# --- Output paths ---
OUTPUT_DIR  = "/kaggle/working/llama3-techqa-qlora"
ADAPTER_DIR = "/kaggle/working/adapter"
MERGED_DIR  = "/kaggle/working/merged-fp16"

# --- Ακολουθια ---
MAX_LENGTH  = 4096       # καλυπτει το max μηκος ΧΩΡΙΣ truncation (επαληθευεται στο cell 5)

# --- LoRA ---
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# --- Training ---
EPOCHS     = 3
LR         = 2e-4        # τυπικο QLoRA· adapters θελουν υψηλοτερο LR απο full-FT
BATCH      = 1
GRAD_ACCUM = 8           # effective batch = BATCH * GRAD_ACCUM = 8
SEED       = 42

## 3. Φορτωση dataset & train/val split (90/10)

In [ ]:
import json
from datasets import Dataset

rows = []
with open(DATASET_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

ABSTAIN = "does not contain a confident solution"
n_b = sum(1 for r in rows if ABSTAIN in r["messages"][-1]["content"])
print("συνολικα παραδειγματα:", len(rows))
print("MODE A (solve):", len(rows) - n_b, " | MODE B (abstain):", n_b,
      f" ({100*n_b/len(rows):.0f}% abstain)")

ds = Dataset.from_list(rows)
split = ds.train_test_split(test_size=0.10, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]
print("\ntrain:", len(train_ds), " | eval:", len(eval_ds))

## 4. Tokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token   # το padding γινεται mask ουτως η αλλως
tokenizer.padding_side = "right"
print("pad:", tokenizer.pad_token, "| eos:", tokenizer.eos_token)

## 5. Επαληθευση μηκους ακολουθιας (πραγματικος tokenizer)

Το `target` βρισκεται στο **τελος** της ακολουθιας. Αν το `MAX_LENGTH` ειναι μικροτερο απο το
πραγματικο μηκος, η right-truncation **κοβει το target** → καταστρεφεται το completion-only SFT.
Αυτο το cell μετραει την πραγματικη κατανομη και επιβεβαιωνει οτι κανενα δειγμα δεν περικοπτεται.

In [ ]:
import numpy as np

def _ntok(msgs):
    out = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=False)
    ids = out["input_ids"] if isinstance(out, dict) else out   # νεες εκδοσεις επιστρεφουν dict
    return len(ids)

lengths = np.array([_ntok(r["messages"]) for r in rows])
for p in [50, 90, 95, 99, 100]:
    print(f"p{p:<3}: {int(np.percentile(lengths, p)):>5} tokens")

over = int((lengths > MAX_LENGTH).sum())
print(f"\nδειγματα > MAX_LENGTH({MAX_LENGTH}): {over} / {len(lengths)} ({100*over/len(lengths):.1f}%)")
if over == 0:
    print("OK — κανενα truncation· το target μενει ακεραιο σε ολα τα δειγματα.")
else:
    print("[ΠΡΟΣΟΧΗ] καποια δειγματα ξεπερνουν το MAX_LENGTH. Αυξησε το MAX_LENGTH,")
    print("η φιλτραρε τα (ΠΟΤΕ right-truncation — κοβει το assistant target).")

## 6. Φορτωση μοντελου σε 4-bit (NF4)

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4: float16 (ΟΧΙ bf16)
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},        # ολο το μοντελο στη GPU 0 (αποφυγη sharding σε 2×T4)
    dtype=torch.float16,       # transformers 5.x: 'dtype' (οχι 'torch_dtype') — αλλιως φορτωνει bf16
)
model.config.use_cache = False   # απαραιτητο με gradient checkpointing
print("Το μοντελο φορτωθηκε σε 4-bit.")

## 7. Διαμορφωση LoRA

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],   # ολες οι γραμμικες προβολες
)

## 8. SFTConfig & SFTTrainer

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_length=MAX_LENGTH,
    assistant_only_loss=True,     # loss ΜΟΝΟ στο assistant — mask system+user (Llama-3 template auto-patched)
    packing=False,                # ασυμβατο με σταθερο masking
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="paged_adamw_8bit",
    fp16=True, bf16=False,        # T4
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    activation_offloading=True,   # κατεβαζει τα activations στη CPU RAM — εξοικονομηση GPU μνημης (T4 16GB)
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    peft_config=peft_config,
    processing_class=tokenizer,
)

## 9. Επαληθευση masking — **πριν** ξοδεψεις GPU quota

Αν το `assistant_only_loss` δεν εφαρμοζεται σωστα (π.χ. το template δεν εχει τα `{% generation %}`
markers), το loss θα υπολογιζοταν πανω σε ολο το prompt — σιωπηρο σφαλμα που χαλαει το fine-tune.
Εδω αποκωδικοποιουμε τα labels (οσα ≠ -100) και επιβεβαιωνουμε οτι περιεχουν **μονο** το target.

In [ ]:
batch = trainer.data_collator([trainer.train_dataset[0]])
input_ids = batch["input_ids"][0]
labels    = batch["labels"][0]

unmasked = [int(t) for t, l in zip(input_ids, labels) if int(l) != -100]
target_txt = tokenizer.decode(unmasked)

print("=== ΜΗ-masked tokens (πανω σε αυτα υπολογιζεται το loss) ===")
print(repr(target_txt))

assert ("Root cause" in target_txt) or ("does not contain" in target_txt), \
    "Το assistant target ΔΕΝ εμφανιζεται στα labels!"
assert "expert IBM technical support" not in target_txt, \
    "ΣΦΑΛΜΑ: το system prompt ΔΕΝ εγινε mask — το assistant_only_loss δεν δουλευει!"
print("\nOK — masking σωστο: loss μονο στο assistant target.")

## 10. Fix precision & Training

Πριν το training, κανουμε cast **ολα τα trainable (LoRA) params σε fp32**. Με fp16 mixed-precision
στον T4, τα master weights πρεπει να ειναι fp32· αν μεινουν bf16, ο fp16 GradScaler σκαει με
`_amp_foreach_non_finite_check_and_unscale_cuda not implemented for BFloat16`. Ο κανονας:
βαση παγωμενη σε 4-bit, adapters fp32, compute fp16 μεσω autocast.

In [ ]:
# Cast trainable (LoRA) params -> fp32 (bulletproof fix για τον fp16 GradScaler στον T4)
n = 0
for _name, p in trainer.model.named_parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.float()
        n += 1
print(f"cast {n} trainable tensors -> fp32")

# επιβεβαιωση: κανενα trainable param δεν ειναι bf16
bad = [nm for nm, p in trainer.model.named_parameters()
       if p.requires_grad and p.dtype == torch.bfloat16]
assert not bad, f"Υπαρχουν ακομα bf16 trainable params: {bad[:3]}"
print("OK — ολα τα trainable params ειναι fp32.")

In [ ]:
train_result = trainer.train()
print(train_result.metrics)

In [ ]:
# Μετρικες αξιολογησης στο val set
print(trainer.evaluate())

## 11. Αποθηκευση adapter

In [ ]:
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("adapter ->", ADAPTER_DIR)
!ls -lh {ADAPTER_DIR}

## 12. Merge adapter → fp16

Απελευθερωνουμε το 4-bit μοντελο, ξαναφορτωνουμε τη βαση σε **fp16 στη CPU** (8B ≈ 16 GB —
χωρανε στη RAM του Kaggle ~29 GB) και κανουμε merge. Το αποτελεσμα ειναι ενα ενιαιο fp16 μοντελο
ετοιμο για GGUF.

In [ ]:
import gc, torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

# απελευθερωση μνημης (αν υπαρχουν ακομα απο το training· ασφαλες και μετα απο restart)
for _n in ["trainer", "model", "merged", "merged4"]:
    if _n in globals():
        del globals()[_n]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.float16, low_cpu_mem_usage=True,
)   # χωρις device_map -> CPU
merged = PeftModel.from_pretrained(base, ADAPTER_DIR)
merged = merged.merge_and_unload()
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print("merged fp16 ->", MERGED_DIR)

del base, merged
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 13. Μετατροπη σε GGUF & κβαντισμος Q4_K_M (llama.cpp)

Παραγει το `.gguf` που φορτωνεται στο Ollama. **Προαιρετικο στο Kaggle** — μπορεις να το τρεξεις
και **τοπικα** στο μηχανημα σου (WSL2 + RTX 3060): κατεβασε μονο τον φακελο `merged-fp16/`
(η ακομα μονο τον adapter, ~150 MB) και τρεξε τα ιδια βηματα.

In [ ]:
# clone llama.cpp + minimal deps (ΟΧΙ ολο το requirements.txt — κατεβαζει CPU torch & μολυνει το env)
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/working/llama.cpp 2>/dev/null
!pip install -q gguf sentencepiece

# Patch tokenizer_config: το transformers 5.x γραφει tokenizer_class="TokenizersBackend"
# που ο converter του llama.cpp δεν φορτωνει. Το θετουμε σε universally-αναγνωρισιμη κλαση.
import json, os
_cfg = os.path.join(MERGED_DIR, "tokenizer_config.json")
with open(_cfg, encoding="utf-8") as f:
    _tc = json.load(f)
_old = _tc.get("tokenizer_class")
_tc["tokenizer_class"] = "PreTrainedTokenizerFast"
with open(_cfg, "w", encoding="utf-8") as f:
    json.dump(_tc, f, ensure_ascii=False)
print(f"tokenizer_class: {_old} -> PreTrainedTokenizerFast")

# fp16 HF -> GGUF f16
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} \
    --outfile /kaggle/working/llama3-techqa-f16.gguf --outtype f16

In [ ]:
# build μονο του llama-quantize (CPU build, γρηγορο)
!cmake -S /kaggle/working/llama.cpp -B /kaggle/working/llama.cpp/build \
    -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /tmp/cmake.log 2>&1
!cmake --build /kaggle/working/llama.cpp/build --target llama-quantize -j2 > /tmp/build.log 2>&1

# f16 -> Q4_K_M
!/kaggle/working/llama.cpp/build/bin/llama-quantize \
    /kaggle/working/llama3-techqa-f16.gguf \
    /kaggle/working/llama3-techqa-Q4_K_M.gguf Q4_K_M

!ls -lh /kaggle/working/*.gguf

## 14. Γρηγορος ποιοτικος ελεγχος (2 held-out tickets)

Φορτωνουμε το merged μοντελο σε **4-bit** (χωραει στα 16 GB του T4) και ελεγχουμε οτι βγαινει
το σωστο format — δομημενη λυση για MODE A, η ακριβως η φραση αποχης για MODE B.

In [ ]:
merged4 = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR, quantization_config=bnb_config, device_map={"": 0},
)
merged4.config.use_cache = True

for r in eval_ds.select(range(2)):
    msgs = r["messages"][:2]   # system + user (χωρις το assistant)
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(0)
    out = merged4.generate(**inputs, max_new_tokens=256, do_sample=False,
                           pad_token_id=tokenizer.eos_token_id)
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print("GOLD:", r["messages"][-1]["content"][:140].replace("\n", " "))
    print("PRED:", gen[:400].replace("\n", " "))
    print("-" * 70)

## 15. Επομενα βηματα — Ollama & τριπλη αξιολογηση

**1. Κατεβασε** το `llama3-techqa-Q4_K_M.gguf` (Kaggle output, ~4.5 GB) στο μηχανημα σου.

**2. Ollama Modelfile** (ιδιος φακελος με το gguf):

```dockerfile
FROM ./llama3-techqa-Q4_K_M.gguf
PARAMETER num_ctx 8192
PARAMETER temperature 0
PARAMETER stop "<|eot_id|>"
```

> **Μη** βαλεις `SYSTEM` στο Modelfile: το `/resolve` του `main.py` στελνει ηδη το ιδιο
> `RAG_SYSTEM_PROMPT` που ειδε το μοντελο στο training — ετσι διατηρειται η συνεπεια train/inference.

```bash
ollama create techqa-llama3 -f Modelfile
```

**3. Τριπλη συγκριση** με το ιδιο `evaluate_generation.py` (n=160), αλλαζοντας μονο το μοντελο/gate:

| Setup | `OLLAMA_MODEL` | Gate |
|---|---|---|
| **base** | `llama3:8b-instruct-q4_K_M` | off |
| **gate** | `llama3:8b-instruct-q4_K_M` | `RAG_GATE_THRESHOLD=5.0` |
| **fine-tuned** | `techqa-llama3` | off |

Αναμενομενο: το fine-tuned μοντελο βελτιωνει το **correct-abstention** και μειωνει το **DEV_Q000
hazard** (answered-when-GT-missing) περισσοτερο απ' οτι το heuristic gate, **χωρις** να θυσιαζει
faithfulness — γιατι διδαχθηκε *συμπεριφορα* αποχης, οχι απλα ενα κατωφλι εμπιστοσυνης.